# 🚗 License Plate Recognition — Adaptive Image Processing Pipeline

| Function | Member | Stage | Adaptive Behaviour |
|---|---|---|---|
| `m1_edge_detection` | M1 | Grayscale + CLAHE + Bilateral + Canny | Auto-tunes Canny thresholds via Otsu; CLAHE for dark/bright images |
| `m2_plate_candidate` | M2 | Morph Closing + Contour Filtering | Multi-pass search with relaxing constraints; scoring by position+area+aspect |
| `m3_perspective_binarize` | M3 | Perspective Correction + Binarization | Picks Otsu vs Adaptive based on local contrast; dynamic output size |


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
import ipywidgets as widgets
import os, warnings
warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

##1: Adaptive Grayscale + Bilateral Filter + Edge Detection

In [ ]:
def m1_edge_detection(image_path, visualize=True):
    """
    M1: Grayscale -> CLAHE -> Bilateral filter -> Auto Canny.

    Adaptive logic:
      - CLAHE clip limit scales with image std-dev: low contrast -> higher clip.
      - Bilateral diameter and sigma scale with image resolution and contrast.
      - Canny thresholds derived from Otsu value of blurred image (sigma rule).

    Returns: original (BGR), gray (CLAHE), edges (binary), meta (dict)
    """
    original = cv2.imread(image_path)
    if original is None:
        raise FileNotFoundError(f'Cannot open image: {image_path}')

    h, w = original.shape[:2]
    gray_raw = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

    # CLAHE: normalise exposure (handles dark carparks, glare, night shots)
    std = float(np.std(gray_raw))
    clip = max(1.0, min(4.0, 80.0 / (std + 1e-5)))
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    gray = clahe.apply(gray_raw)

    # Bilateral: diameter proportional to image size
    bil_d = max(5, min(15, int(min(h, w) / 60) * 2 + 1))
    sigma = max(15, min(75, int(std * 0.8)))
    filtered = cv2.bilateralFilter(gray, bil_d, sigma, sigma)

    # Auto Canny: derive thresholds from Otsu on blurred image
    blurred = cv2.GaussianBlur(filtered, (5, 5), 0)
    otsu_val, _ = cv2.threshold(blurred, 0, 255,
                                cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    canny_low  = max(10,  min(int(otsu_val * 0.5), 100))
    canny_high = max(80,  min(int(otsu_val * 1.2), 350))
    edges = cv2.Canny(filtered, canny_low, canny_high)

    meta = dict(clahe_clip=round(clip, 2), bilateral_d=bil_d,
                sigma=sigma, canny=[canny_low, canny_high],
                img_std=round(std, 1))

    if visualize:
        fig, axes = plt.subplots(1, 4, figsize=(20, 4))
        axes[0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
        axes[0].set_title(f'Original  {w}x{h}')
        axes[1].imshow(gray_raw, cmap='gray')
        axes[1].set_title('Grayscale')
        axes[2].imshow(gray, cmap='gray')
        axes[2].set_title(f'CLAHE  clip={clip:.1f}  std={std:.0f}')
        axes[3].imshow(edges, cmap='gray')
        axes[3].set_title(f'Canny [{canny_low}, {canny_high}]')
        for ax in axes: ax.axis('off')
        plt.suptitle('M1 — Edge Detection', fontweight='bold', y=1.02)
        plt.tight_layout()
        plt.show()
        print(f'  [M1] {meta}')

    return original, gray, edges, meta

## 2: Adaptive Morph Closing + Contour Filtering

In [ ]:
def m2_plate_candidate(original, edges, visualize=True):
    """
    M2: Morph closing -> contour filter -> best plate bounding box.

    Adaptive logic:
      - Kernel width ~2% of image width (resolution-independent).
      - 3 passes: strict -> medium -> loose aspect/area constraints.
      - Candidates scored by area * aspect_fitness * vertical_position.
      - Bounding box padded 2% each side to avoid clipping characters.

    Returns: plate_crop (BGR), best_contour, best_rect (x,y,w,h), meta (dict)
    """
    img_h, img_w = edges.shape
    img_area = img_h * img_w

    kw = max(9, int(img_w * 0.022))
    if kw % 2 == 0: kw += 1

    passes = [
        dict(aspect=(2.5, 6.5), area=(0.0008, 0.045)),
        dict(aspect=(1.8, 8.0), area=(0.0004, 0.07)),
        dict(aspect=(1.5, 10.), area=(0.0002, 0.10)),
    ]

    kernel  = cv2.getStructuringElement(cv2.MORPH_RECT, (kw, 1))
    closed  = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
    dilated = cv2.dilate(closed, np.ones((3, 3), np.uint8), iterations=1)

    best_contour = None
    best_rect    = None
    best_score   = -1
    pass_used    = 0

    for p_idx, p in enumerate(passes):
        src = closed if p_idx < 2 else dilated
        contours, _ = cv2.findContours(src, cv2.RETR_EXTERNAL,
                                        cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            area   = w * h
            aspect = w / float(h) if h > 0 else 0
            if not (p['aspect'][0] <= aspect <= p['aspect'][1]): continue
            if not (p['area'][0] * img_area <= area <= p['area'][1] * img_area): continue
            aspect_fitness = 1.0 - abs(aspect - 4.0) / 4.0
            y_center = (y + h / 2) / img_h
            pos_weight = 1.0 + 0.3 * (y_center > 0.35)
            score = (area / img_area) * aspect_fitness * pos_weight
            if score > best_score:
                best_score = score; best_contour = cnt
                best_rect = (x, y, w, h); pass_used = p_idx + 1
        if best_contour is not None: break

    if best_contour is None:
        print('[M2] No candidate found — using full image.')
        best_rect = (0, 0, img_w, img_h)
        best_contour = np.array([[[0,0]],[[img_w,0]],[[img_w,img_h]],[[0,img_h]]])
        pass_used = -1

    x, y, w, h = best_rect
    pad_x = max(4, int(w * 0.02))
    pad_y = max(4, int(h * 0.05))
    x1 = max(0, x - pad_x);       y1 = max(0, y - pad_y)
    x2 = min(img_w, x + w + pad_x); y2 = min(img_h, y + h + pad_y)
    best_rect = (x1, y1, x2 - x1, y2 - y1)
    plate_crop = original[y1:y2, x1:x2]

    meta = dict(pass_used=pass_used, kernel_w=kw,
                score=round(best_score, 4), rect=best_rect)

    if visualize:
        debug = original.copy()
        rx, ry, rw, rh = best_rect
        cv2.rectangle(debug, (rx, ry), (rx+rw, ry+rh), (0, 220, 60), 3)
        lbl = f'Pass {pass_used}  score={best_score:.3f}'
        cv2.putText(debug, lbl, (rx, max(ry-8, 20)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,220,60), 2)
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        axes[0].imshow(closed, cmap='gray')
        axes[0].set_title(f'Morph Close  kernel=({kw},1)')
        axes[1].imshow(cv2.cvtColor(debug, cv2.COLOR_BGR2RGB))
        axes[1].set_title('Best Candidate')
        axes[2].imshow(cv2.cvtColor(plate_crop, cv2.COLOR_BGR2RGB))
        axes[2].set_title(f'Cropped  {rw}x{rh}px')
        for ax in axes: ax.axis('off')
        plt.suptitle('M2 — Plate Candidate', fontweight='bold', y=1.02)
        plt.tight_layout(); plt.show()
        print(f'  [M2] {meta}')

    return plate_crop, best_contour, best_rect, meta

## 3: Adaptive Perspective Correction + Binarization

In [ ]:
def m3_perspective_binarize(original, best_rect, visualize=True):
    """
    M3: Perspective warp -> adaptive binarization.

    Adaptive logic:
      - Tries multiple approxPolyDP epsilons to get a 4-corner quad;
        falls back to bounding rect if none found.
      - Output width derived from detected aspect ratio (not fixed 320x100).
      - Auto-selects Otsu (high contrast) vs Adaptive Gaussian (low/uneven
        contrast) based on local std-dev of warped crop.
      - Median blur after threshold removes salt-and-pepper noise.

    Returns: binary_plate (ndarray), warped_gray (ndarray), meta (dict)
    """
    gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
    x, y, w, h = best_rect

    # Try to find a quadrilateral inside the crop
    crop_gray = gray[y:y+h, x:x+w]
    _, crop_edges = cv2.threshold(
        cv2.GaussianBlur(crop_gray, (3,3), 0), 0, 255,
        cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    crop_cnts, _ = cv2.findContours(crop_edges, cv2.RETR_EXTERNAL,
                                     cv2.CHAIN_APPROX_SIMPLE)

    quad = None
    if crop_cnts:
        biggest = max(crop_cnts, key=cv2.contourArea)
        peri = cv2.arcLength(biggest, True)
        for eps in [0.02, 0.03, 0.05, 0.08]:
            approx = cv2.approxPolyDP(biggest, eps * peri, True)
            if len(approx) == 4:
                quad = approx.reshape(4, 2).astype(np.float32)
                quad[:, 0] += x; quad[:, 1] += y
                break

    def _order(pts):
        rect = np.zeros((4, 2), dtype=np.float32)
        s = pts.sum(axis=1); d = np.diff(pts, axis=1).ravel()
        rect[0] = pts[np.argmin(s)]; rect[2] = pts[np.argmax(s)]
        rect[1] = pts[np.argmin(d)]; rect[3] = pts[np.argmax(d)]
        return rect

    src_pts = _order(quad) if quad is not None else \
              np.array([[x,y],[x+w,y],[x+w,y+h],[x,y+h]], dtype=np.float32)
    corner_src = 'approxPolyDP' if quad is not None else 'bounding rect'

    out_h = 80
    out_w = max(160, min(480, int(out_h * (w / float(h) if h > 0 else 4.0))))
    dst_pts = np.array([[0,0],[out_w-1,0],[out_w-1,out_h-1],[0,out_h-1]],
                        dtype=np.float32)
    M = cv2.getPerspectiveTransform(src_pts, dst_pts)
    warped = cv2.warpPerspective(gray, M, (out_w, out_h))

    local_std = float(np.std(warped))
    if local_std >= 30:
        _, binary = cv2.threshold(warped, 0, 255,
                                   cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        method = 'Otsu'
    else:
        block = max(11, int(out_w / 15) | 1)
        binary = cv2.adaptiveThreshold(warped, 255,
                    cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, block, 4)
        method = f'Adaptive(block={block})'

    if np.mean(binary) < 127:
        binary = cv2.bitwise_not(binary)
    binary = cv2.medianBlur(binary, 3)

    meta = dict(corner_src=corner_src, out_size=(out_w, out_h),
                local_std=round(local_std, 1), binarize=method)

    if visualize:
        fig, axes = plt.subplots(1, 3, figsize=(16, 4))
        axes[0].imshow(cv2.cvtColor(original[y:y+h, x:x+w], cv2.COLOR_BGR2RGB))
        axes[0].set_title('Plate Crop (from M2)')
        axes[1].imshow(warped, cmap='gray')
        axes[1].set_title(f'Warped  {out_w}x{out_h}  [{corner_src}]')
        axes[2].imshow(binary, cmap='gray')
        axes[2].set_title(f'Binary [{method}]  std={local_std:.0f}')
        for ax in axes: ax.axis('off')
        plt.suptitle('M3 — Perspective + Binarization', fontweight='bold', y=1.02)
        plt.tight_layout(); plt.show()
        print(f'  [M3] {meta}')

    return binary, warped, meta

## Combined Pipeline

In [ ]:
def run_pipeline(image_path, visualize=True):
    """Run M1 -> M2 -> M3. Returns (binary_plate, all_meta)."""
    sep = '─' * 55
    print(sep)
    print('  M1  Grayscale + CLAHE + Bilateral + Auto-Canny')
    print(sep)
    original, gray, edges, m1 = m1_edge_detection(image_path, visualize=visualize)

    print(sep)
    print('  M2  Morph Closing + Multi-pass Contour Filter')
    print(sep)
    plate_crop, contour, best_rect, m2 = m2_plate_candidate(
        original, edges, visualize=visualize)

    print(sep)
    print('  M3  Perspective Warp + Adaptive Binarization')
    print(sep)
    binary, warped, m3 = m3_perspective_binarize(
        original, best_rect, visualize=visualize)

    all_meta = {'M1': m1, 'M2': m2, 'M3': m3}
    print('\n✅  Pipeline complete')
    for k, v in all_meta.items():
        print(f'   [{k}] {v}')

    return binary, all_meta

## Interactive Widget — Upload & Run

In [ ]:
upload  = widgets.FileUpload(accept='image/*', multiple=False, description='Upload')
run_btn = widgets.Button(description='Run Pipeline', button_style='success')
out     = widgets.Output()

def _on_run(b):
    out.clear_output(wait=True)
    with out:
        if not upload.value:
            print('Please upload an image first.'); return
        fname = list(upload.value.keys())[0]
        data  = upload.value[fname]['content']
        path  = f'/tmp/{fname}'
        with open(path, 'wb') as f: f.write(bytes(data))
        try:
            run_pipeline(path, visualize=True)
        except Exception as e:
            print(f'Error: {e}')

run_btn.on_click(_on_run)
display(widgets.HBox([upload, run_btn]), out)

## Quick Test — Run on a Local File Path

In [ ]:
IMAGE_PATH = 'car.jpg'   # <-- change to your image

if os.path.exists(IMAGE_PATH):
    binary_result, meta = run_pipeline(IMAGE_PATH, visualize=True)
else:
    print(f'File not found: {IMAGE_PATH}')
    print('Set IMAGE_PATH to a valid image in this folder.')